<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
!pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
clients = f"read_parquet('{rel}/dim_clients.parquet')"
content = f"read_parquet('{rel}/dim_content.parquet')"
daily = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"
q90 = f"read_parquet('{rel}/fact_content_query_90d.parquet')"
month = "2026-03"
daily_m = f"read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')"
print(con.sql(f"SELECT COUNT(*) AS clients FROM {clients}").df().to_string(index=False))
print(con.sql(f"SELECT COUNT(*) AS rows_month FROM {daily_m}").df().to_string(index=False))

 clients
     104
 rows_month
    9841378


In [26]:
print(con.sql(f"DESCRIBE SELECT * FROM {daily_m}").df().to_string(index=False))
print()
print(con.sql(f"DESCRIBE SELECT * FROM {content}").df().to_string(index=False))
print()
print(con.sql(f"DESCRIBE SELECT * FROM {clients}").df().to_string(index=False))

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

## 1. Unit of analysis + time window

One row = one page, for one client, aggregated over March 2026.

The raw table is finer than that. fact_content_daily_performance is page-day: one row per report_date, client and content item. My analysis unit is the page-month, which I build by aggregating that month's daily rows up to one row per content_hash_id within a client_hash_id.

Window: month=2026-03, a single calendar month. I picked a mid-panel month deliberately. The card warns that the _sample table is June 2026, the final month, which is the natural outcome window for any past-to-future label, so I treat it as a sealed test set and never iterate on it.

One row therefore means: this page, on this client's site, over these 31 days. Impressions and clicks sum across the month. Position does not sum, which is why the table carries gsc_sum_position alongside the daily average.

In [27]:
print(con.sql(f"""
SELECT MIN(report_date) AS first_day, MAX(report_date) AS last_day, COUNT(DISTINCT report_date) AS days
FROM {daily_m}
""").df().to_string(index=False))
print(con.sql(f"""
SELECT COUNT(*) AS rows_with_duplicate_grain FROM (
  SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
  FROM {daily_m} GROUP BY 1,2,3 HAVING c > 1
)
""").df().to_string(index=False))
print(con.sql(f"""
SELECT COUNT(*) AS daily_rows, COUNT(DISTINCT content_hash_id) AS pages, COUNT(DISTINCT client_hash_id) AS clients
FROM {daily_m}
""").df().to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 first_day   last_day  days
2026-03-01 2026-03-31    31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rows_with_duplicate_grain
                         0
 daily_rows  pages  clients
    9841378 331437       55


## 2. Fields: feature / label / context / excluded

Label. gsc_clicks, and CTR derived from it. CTR is clicks over impressions, both observed counts.

Context, never a model feature. gsc_impressions sits here deliberately. It is the denominator of my label, so letting the model learn from it would be learning half the answer. I use it only for the volume floor and for weighting. Also here: client_hash_id, content_hash_id, report_date, month. IDs are for grouping and splitting, not for learning.

Features, knowable before the click happens. gsc_avg_position, word_count, char_count, search_volume, competition, cpc, backlinks, category_count, content_type, main_intent, keyword_token_count, url_char_count, and page age computed from content_created_date.

Excluded, each with a why.

All ga4_* fields and every sessions_* split: these describe what someone did after arriving. I am predicting whether they click at all, so this is future information relative to my decision moment.
scroll_events: same reason, post-click behaviour.
All ai_* fields: post-click, and the dataset card notes AI session data is desperately sparse.
last_optimized_date and optimization_eligible_date: product decision flags. Someone already decided this page needed work, so these encode the answer I am trying to produce.
provider_used, model_used: decisions about how the content was generated, not properties of the page a searcher sees.
is_deleted: filter, not signal.

What these numbers changed. GSC availability sits at 36.7% of March rows, and my label is a GSC field, so nearly two thirds of the raw rows cannot carry a label. GA4 availability is 4.2%, which means excluding GA4 features was going to happen whether I wanted it or not. AI session fields appear on 0.06% of rows. And 19.5% of pages in dim_content are flagged deleted, so is_deleted is a filter I have to apply rather than a rare edge case.

Five-feature frame, knowable at the decision moment.

gsc_avg_position — knowable because it describes where the page already sits when someone searches, before any click happens.
word_count — knowable because it is a property of the page as published, fixed before the search occurs.
search_volume — knowable because it describes the keyword's demand, measured independently of this page's performance.
competition — knowable because it describes the keyword market, not this page's outcome.
backlinks — knowable because links pointing at the page exist before the impression is served.

Each of these exists at the moment the page is shown in results. None of them is computed from clicks.

In [28]:
print(con.sql(f"""
SELECT
  COUNT(*) AS pages_total,
  SUM(CASE WHEN last_optimized_date IS NOT NULL THEN 1 ELSE 0 END) AS has_last_optimized,
  SUM(CASE WHEN is_published THEN 1 ELSE 0 END) AS published,
  SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS deleted
FROM {content}
""").df().to_string(index=False))
print(con.sql(f"""
SELECT
  AVG(CASE WHEN sessions_ai > 0 THEN 1.0 ELSE 0 END) AS share_rows_with_ai_sessions,
  AVG(CASE WHEN ga4_data_available THEN 1.0 ELSE 0 END) AS share_ga4_available,
  AVG(CASE WHEN gsc_data_available THEN 1.0 ELSE 0 END) AS share_gsc_available
FROM {daily_m}
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 pages_total  has_last_optimized  published  deleted
      519606             45396.0   411540.0 101559.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 share_rows_with_ai_sessions  share_ga4_available  share_gsc_available
                    0.000562             0.042064             0.366926


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim in sections 1 and 2 gets a query here.

Q1, grain. Aggregating the daily rows to one row per client and page returns zero duplicates, so the page-month unit I stated in section 1 holds.

Q2, slice size and span. 9,841,378 daily rows across 331,437 pages, spanning 2026-03-01 to 2026-03-31. That is about 29.7 rows per page against 31 days, so pages do not all appear every day.

Q3, availability. Filtering gsc_data_available IS TRUE leaves 3,611,061 rows on 176,738 pages, 36.7% of the slice and 53.3% of the pages. This is the population my label can exist for at all. Anything I say about CTR describes those pages, not the 9.8M rows the month appears to hold.

The aggregate CTR across that filtered slice is 821,832 clicks over 280,657,589 impressions, which is 0.29%. That matches the median CTR I measured on the starter dataset in w02 almost exactly, which is a useful sanity check that the two datasets describe the same world.

In [29]:
print("Q1 grain after aggregation to page-month")
print(con.sql(f"""
SELECT COUNT(*) AS duplicate_page_rows FROM (
  SELECT client_hash_id, content_hash_id, COUNT(*) c
  FROM (SELECT DISTINCT client_hash_id, content_hash_id FROM {daily_m})
  GROUP BY 1,2 HAVING c > 1
)
""").df().to_string(index=False))
print("\nQ2 slice size and date span")
print(con.sql(f"""
SELECT COUNT(*) AS daily_rows, COUNT(DISTINCT content_hash_id) AS pages,
       MIN(report_date) AS first_day, MAX(report_date) AS last_day
FROM {daily_m}
""").df().to_string(index=False))
print("\nQ3 how many rows survive the availability filter")
print(con.sql(f"""
SELECT COUNT(*) AS rows_gsc_true, COUNT(DISTINCT content_hash_id) AS pages_gsc_true,
       SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
FROM {daily_m} WHERE gsc_data_available IS TRUE
""").df().to_string(index=False))


Q1 grain after aggregation to page-month
 duplicate_page_rows
                   0

Q2 slice size and date span
 daily_rows  pages  first_day   last_day
    9841378 331437 2026-03-01 2026-03-31

Q3 how many rows survive the availability filter


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rows_gsc_true  pages_gsc_true  impressions   clicks
       3611061          176738  280657589.0 821832.0


## 4. Data limits

The panel is unbalanced. Client GSC histories start anywhere from 2025-01-27 to 2026-06-02. A client whose data begins in May 2026 has no March 2026 rows at all, and their absence is not a zero. It means the client was not being observed yet. Any trend I compute across months is measuring a changing set of clients as much as changing performance.

A quarter of clients can never appear. 26 of 104 have no_search_or_analytics_access, and a further 10 are source_only_missing_client_dimension. Only 47 clients survive into my filtered March slice. So this dataset cannot tell me anything about roughly half the client base.

One client dominates. Median 1,472 pages per client, maximum 27,425. That is nearly 19 times the median. Any pattern I find could be one client's site conventions rather than something general, so a per-client check is mandatory before I trust any result.

GA4 is effectively unusable. 53 clients have no ga4_data_start at all, and GA4 is available on 4.2% of March rows. Engagement questions are out of reach with this data.

What this data can never tell me. It cannot tell me whether a page changed because of anything anyone did. There are dates on the pages but the fact table records outcomes, not interventions, and I have no record of what was edited when. It cannot support a causal claim about ranking. And it cannot tell me about pages that were never indexed, since a page with no impressions simply produces no row.

In [30]:
print("client history start dates")
print(con.sql(f"""
SELECT MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest,
       COUNT(*) AS clients, SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS no_ga4_start
FROM {clients}
""").df().to_string(index=False))
print("\nclients per access profile")
print(con.sql(f"SELECT access_profile, COUNT(*) n FROM {clients} GROUP BY 1 ORDER BY n DESC").df().to_string(index=False))
print("\npages per client in the filtered March slice")
print(con.sql(f"""
SELECT MEDIAN(pages) AS median_pages, MAX(pages) AS max_pages, COUNT(*) AS clients FROM (
  SELECT client_hash_id, COUNT(DISTINCT content_hash_id) AS pages
  FROM {daily_m} WHERE gsc_data_available IS TRUE GROUP BY 1
)
""").df().to_string(index=False))

client history start dates
  earliest     latest  clients  no_ga4_start
2025-01-27 2026-06-02      104          53.0

clients per access profile
                      access_profile  n
                         gsc_and_ga4 53
       no_search_or_analytics_access 26
                            gsc_only 14
source_only_missing_client_dimension 10
                            ga4_only  1

pages per client in the filtered March slice
 median_pages  max_pages  clients
       1472.0      27425       47


The trap, run on purpose.

I added clicks / impressions as a feature. My label is CTR below 0.1, and CTR is clicks over impressions. So the feature is the label with the threshold removed. Held-out AUC came back at 1.000 on 101,441 pages.

A perfect held-out score is not a good result. It means the model has been handed the answer and asked to recognise it. There is no world in which a real signal separates 101,441 pages flawlessly.

Removing that one column drops AUC to 0.765 against a base rate of 0.468. That number is worth something. The first is worth nothing.

This is the same failure I saw in starter notebook 02, where a tree given a rule-derived label rediscovered the rule's threshold and scored a perfect 1.000. Causing it deliberately is more useful than reading about it, because the tell is the score itself. Anything near-perfect on held-out data means checking what went into the features before celebrating.

Deleted and not kept. The honest number for this slice is 0.765.

In [31]:
lab = con.sql(f"""
SELECT content_hash_id,
       SUM(gsc_clicks) AS clicks,
       SUM(gsc_impressions) AS impressions,
       AVG(gsc_avg_position) AS avg_position
FROM {daily_m} WHERE gsc_data_available IS TRUE
GROUP BY 1 HAVING SUM(gsc_impressions) >= 100
""").df()
lab["ctr"] = lab["clicks"] / lab["impressions"] * 100
lab["underperforming"] = (lab["ctr"] < 0.1).astype(int)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
def run(cols, name):
    X = lab[cols].fillna(-1)
    y = lab["underperforming"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    m = RandomForestClassifier(n_estimators=60, max_depth=6, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    print(f"{name}: held-out AUC = {roc_auc_score(yte, m.predict_proba(Xte)[:, 1]):.3f}")
print(f"pages scored: {len(lab):,}, underperforming share: {lab['underperforming'].mean():.3f}")
lab["leaky_clicks_per_impression"] = lab["clicks"] / lab["impressions"]
run(["avg_position", "impressions", "leaky_clicks_per_impression"], "WITH leaked feature (deleted after this run)")
run(["avg_position", "impressions"], "honest features only")
lab = lab.drop(columns=["leaky_clicks_per_impression"])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages scored: 101,441, underperforming share: 0.468
WITH leaked feature (deleted after this run): held-out AUC = 1.000
honest features only: held-out AUC = 0.763


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.